<a href="https://colab.research.google.com/github/Jun-1112/FYP-project-Trunk-and-weed-detection-for-agricultural-usage/blob/main/Video_Inference_%26_Other_metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Setup, paths, and file checks
!pip install ultralytics -q

import os
import torch
from google.colab import drive

drive.mount('/content/drive')

MODEL_ROOT = "/content/drive/MyDrive/Colab Notebooks/best.pt"
VIDEO_PATH = "/content/drive/MyDrive/Colab Notebooks/fyp_video_sample2.mp4"

MODEL_PATHS = {
    "Weed Detection (YOLOv8n)":        os.path.join(MODEL_ROOT, "weed_det",  "best_weeddet.pt"),
    "Trunk Detection (YOLOv8n)":       os.path.join(MODEL_ROOT, "trunk_det", "best_trunkdet.pt"),
    "Trunk Segmentation (YOLOv8n-seg)": os.path.join(MODEL_ROOT, "trunk_seg", "bestseg.pt"),
}

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

assert os.path.exists(VIDEO_PATH), f"Video not found: {VIDEO_PATH}"
print("Video found:", VIDEO_PATH)

for name, path in MODEL_PATHS.items():
    assert os.path.exists(path), f"Model not found for {name}: {path}"
    print(f"Found: {name}")

In [ ]:
# To train Video inference — annotated output for all three models
from ultralytics import YOLO

RUN_NAMES = {
    "Weed Detection (YOLOv8n)":         "weeddet_inference",
    "Trunk Detection (YOLOv8n)":        "trunkdet_inference",
    "Trunk Segmentation (YOLOv8n-seg)": "trunkseg_inference",
}

for name, path in MODEL_PATHS.items():
    print(f"\n--- Running inference: {name} ---")
    model = YOLO(path)
    model.predict(
        source=VIDEO_PATH,
        conf=0.25,
        device=DEVICE,
        save=True,
        project='/content/runs/inference',
        name=RUN_NAMES[name],
        exist_ok=True,
        verbose=False,
    )
    print(f"Saved to /content/runs/inference/{RUN_NAMES[name]}/")

print("\nAll three inference runs complete.")

In [ ]:
# To convert outputs to playable MP4 and preview
from IPython.display import Video, display

os.makedirs('/content/playable_outputs', exist_ok=True)

for name, run_name in RUN_NAMES.items():
    src_files = glob.glob(f'/content/runs/inference/{run_name}/*.avi') + \
                glob.glob(f'/content/runs/inference/{run_name}/*.mp4')
    if not src_files:
        print(f"No output video found for {run_name}")
        continue

    src = src_files[0]
    dst = f'/content/playable_outputs/{run_name}.mp4'
    !ffmpeg -loglevel error -i "{src}" -vcodec libx264 -y "{dst}"
    print(f"Converted: {run_name}.mp4")

# Preview one (change the filename to view a different model's output)
display(Video('/content/playable_outputs/trunkseg_inference.mp4', embed=True, width=800))

In [ ]:
# To obtain Latency / FPS benchmark (RQ2)
import csv, time, statistics
import cv2
import numpy as np

OUTPUT_DIR = "/content/evaluation_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def benchmark_model(model_name, model_path, video_path, warmup_frames=10):
    print(f"\n--- Benchmarking: {model_name} ---")
    model = YOLO(model_path)

    # Warm-up: exclude first N frames so GPU init/caching doesn't skew timings
    cap = cv2.VideoCapture(video_path)
    for _ in range(warmup_frames):
        ret, frame = cap.read()
        if not ret:
            break
        model.predict(source=frame, imgsz=640, device=DEVICE, verbose=False)
    cap.release()

    cap = cv2.VideoCapture(video_path)
    inference_times, total_times = [], []
    frame_count = 0
    benchmark_start = time.perf_counter()

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        results = model.predict(source=frame, imgsz=640, device=DEVICE, verbose=False)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t1 = time.perf_counter()

        speed = results[0].speed
        inference_times.append(speed.get("inference", 0))
        total_times.append((t1 - t0) * 1000)
        frame_count += 1

    benchmark_duration = time.perf_counter() - benchmark_start
    cap.release()

    mean_total = statistics.mean(total_times)
    p95_total = float(np.percentile(total_times, 95))

    res = {
        "Model": model_name,
        "Frames": frame_count,
        "Mean Inference (ms)": statistics.mean(inference_times),
        "Median Inference (ms)": statistics.median(inference_times),
        "Mean Total Latency (ms)": mean_total,
        "Median Total Latency (ms)": statistics.median(total_times),
        "P95 Total Latency (ms)": p95_total,
        "Mean FPS": 1000 / mean_total,
        "P95 FPS Floor": 1000 / p95_total,
        "Actual Processing FPS": frame_count / benchmark_duration,
    }

    print(f"Frames: {frame_count} | Mean latency: {mean_total:.2f} ms | "
          f"Mean FPS: {res['Mean FPS']:.2f} | Actual FPS: {res['Actual Processing FPS']:.2f}")
    return res

all_results = [benchmark_model(n, p, VIDEO_PATH) for n, p in MODEL_PATHS.items()]

summary_csv = os.path.join(OUTPUT_DIR, "latency_results.csv")
with open(summary_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(all_results[0].keys()))
    writer.writeheader()
    writer.writerows(all_results)

print("\n" + "=" * 78)
print("FINAL LATENCY / FPS COMPARISON")
print("=" * 78)
print(f"{'Model':<35}{'Mean ms':>11}{'P95 ms':>11}{'Mean FPS':>11}{'Actual FPS':>12}")
print("-" * 78)
for r in all_results:
    print(f"{r['Model']:<35}{r['Mean Total Latency (ms)']:>11.2f}"
          f"{r['P95 Total Latency (ms)']:>11.2f}{r['Mean FPS']:>11.2f}"
          f"{r['Actual Processing FPS']:>12.2f}")
print("=" * 78)
print("Saved:", summary_csv)

In [ ]:
# To write compute_iou.py
%%writefile compute_iou.py
"""
Compares detection vs segmentation localisation precision against ground-truth
polygon mask for the trunk class.

Metrics:
  detection_box_iou     = predicted detection box vs ground-truth box
  segmentation_mask_iou = predicted polygon mask vs ground-truth mask
  segmentation_box_iou  = seg model's own bounding box vs ground-truth box
  extraneous_area_ratio =  (det_box_area - seg_mask_area) / det_box_area
"""
import argparse, glob, os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


def load_yolo_seg_labels(label_path, img_w, img_h, class_id):
    """Parse a YOLO-segmentation .txt into (polygon_px, bbox_xyxy) per instance."""
    instances = []
    if not os.path.exists(label_path):
        return instances
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts or int(float(parts[0])) != class_id:
                continue
            pts = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(-1, 2)
            pts[:, 0] *= img_w
            pts[:, 1] *= img_h
            pts = pts.astype(np.int32)
            bbox = (pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max())
            instances.append((pts, bbox))
    return instances


def polygon_to_mask(pts, img_w, img_h):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def box_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter / union if union > 0 else 0.0


def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det_model", required=True)
    ap.add_argument("--seg_model", required=True)
    ap.add_argument("--images", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--trunk_class_id", type=int, default=0)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--out_csv", default="iou_results.csv")
    args = ap.parse_args()

    det_model, seg_model = YOLO(args.det_model), YOLO(args.seg_model)
    rows = []

    for img_path in sorted(glob.glob(os.path.join(args.images, "*.*"))):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]

        gt_instances = load_yolo_seg_labels(
            os.path.join(args.labels, stem + ".txt"), w, h, args.trunk_class_id)
        if not gt_instances:
            continue

        det_res = det_model(img, verbose=False, conf=args.conf)[0]
        det_boxes = [b for b, c in zip(det_res.boxes.xyxy.cpu().numpy(),
                                       det_res.boxes.cls.cpu().numpy())
                     if int(c) == args.trunk_class_id] if det_res.boxes is not None else []

        seg_res = seg_model(img, verbose=False, conf=args.conf)[0]
        seg_masks, seg_boxes = [], []
        if seg_res.masks is not None and seg_res.boxes is not None:
            mask_data = seg_res.masks.data.cpu().numpy()
            for i, (b, c) in enumerate(zip(seg_res.boxes.xyxy.cpu().numpy(),
                                           seg_res.boxes.cls.cpu().numpy())):
                if int(c) == args.trunk_class_id:
                    m = cv2.resize(mask_data[i], (w, h),
                                   interpolation=cv2.INTER_NEAREST).astype(np.uint8)
                    seg_masks.append(m)
                    seg_boxes.append(b)

        for gt_pts, gt_box in gt_instances:
            gt_mask = polygon_to_mask(gt_pts, w, h)

            best_det_iou, best_det_box = 0.0, None
            for db in det_boxes:
                iou = box_iou(db, gt_box)
                if iou > best_det_iou:
                    best_det_iou, best_det_box = iou, db

            best_mask_iou, best_seg_mask = 0.0, None
            best_seg_box_iou = 0.0
            for sb, sm in zip(seg_boxes, seg_masks):
                if mask_iou(sm, gt_mask) > best_mask_iou:
                    best_mask_iou, best_seg_mask = mask_iou(sm, gt_mask), sm
                best_seg_box_iou = max(best_seg_box_iou, box_iou(sb, gt_box))

            extraneous = None
            if best_det_box is not None and best_seg_mask is not None:
                det_area = (best_det_box[2]-best_det_box[0]) * (best_det_box[3]-best_det_box[1])
                if det_area > 0:
                    extraneous = (det_area - best_seg_mask.sum()) / det_area

            rows.append({
                "frame": stem,
                "detection_box_iou": round(best_det_iou, 4),
                "segmentation_mask_iou": round(best_mask_iou, 4),
                "segmentation_box_iou": round(best_seg_box_iou, 4),
                "extraneous_area_ratio": round(extraneous, 4) if extraneous is not None else None,
            })

    df = pd.DataFrame(rows)
    df.to_csv(args.out_csv, index=False)

    print(f"\nEvaluated {len(df)} ground-truth trunk instances.\n")
    print("=== Mean Results ===")
    print(f"Detection Box IoU (vs GT box):               {df['detection_box_iou'].mean():.4f}")
    print(f"Segmentation Mask IoU (vs GT mask):          {df['segmentation_mask_iou'].mean():.4f}")
    print(f"Segmentation Box IoU (vs GT box):            {df['segmentation_box_iou'].mean():.4f}")
    print(f"Extraneous Area Ratio (det box vs seg mask): {df['extraneous_area_ratio'].mean():.4f}")
    print(f"\nSaved per-instance results to {args.out_csv}")


if __name__ == "__main__":
    main()

In [ ]:
# To run IoU evaluation (RQ1)
!python compute_iou.py \
  --det_model "/content/drive/MyDrive/Colab Notebooks/best.pt/trunk_det/best_trunkdet.pt" \
  --seg_model "/content/drive/MyDrive/Colab Notebooks/best.pt/trunk_seg/bestseg.pt" \
  --images "/content/gt_check/images/Train" \
  --labels "/content/gt_check/labels/Train" \
  --trunk_class_id 0 \
  --out_csv /content/evaluation_outputs/iou_results.csv

In [ ]:
# To write robustness_stratify.py
%%writefile robustness_stratify.py
""" Stratifies annotated frames into Low/Medium/High motion-blur and lighting
groups and reports mean IoU per group for both models.

Blur proxy     : Laplacian variance (lower = blurrier)
Lighting proxy : mean grayscale pixel brightness """
import argparse, glob, os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO


def load_yolo_seg_labels(label_path, img_w, img_h, class_id):
    instances = []
    if not os.path.exists(label_path):
        return instances
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if not parts or int(float(parts[0])) != class_id:
                continue
            pts = np.array(list(map(float, parts[1:])), dtype=np.float32).reshape(-1, 2)
            pts[:, 0] *= img_w
            pts[:, 1] *= img_h
            pts = pts.astype(np.int32)
            bbox = (pts[:, 0].min(), pts[:, 1].min(), pts[:, 0].max(), pts[:, 1].max())
            instances.append((pts, bbox))
    return instances


def polygon_to_mask(pts, img_w, img_h):
    mask = np.zeros((img_h, img_w), dtype=np.uint8)
    cv2.fillPoly(mask, [pts], 1)
    return mask


def box_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    union = (b1[2]-b1[0])*(b1[3]-b1[1]) + (b2[2]-b2[0])*(b2[3]-b2[1]) - inter
    return inter / union if union > 0 else 0.0


def mask_iou(m1, m2):
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return inter / union if union > 0 else 0.0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--det_model", required=True)
    ap.add_argument("--seg_model", required=True)
    ap.add_argument("--images", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--trunk_class_id", type=int, default=0)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--out_csv", default="robustness_results.csv")
    args = ap.parse_args()

    det_model, seg_model = YOLO(args.det_model), YOLO(args.seg_model)
    rows = []

    for img_path in sorted(glob.glob(os.path.join(args.images, "*.*"))):
        stem = os.path.splitext(os.path.basename(img_path))[0]
        img = cv2.imread(img_path)
        if img is None:
            continue
        h, w = img.shape[:2]
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        brightness_mean = gray.mean()

        gt_instances = load_yolo_seg_labels(
            os.path.join(args.labels, stem + ".txt"), w, h, args.trunk_class_id)
        if not gt_instances:
            continue

        det_res = det_model(img, verbose=False, conf=args.conf)[0]
        det_boxes = [b for b, c in zip(det_res.boxes.xyxy.cpu().numpy(),
                                       det_res.boxes.cls.cpu().numpy())
                     if int(c) == args.trunk_class_id] if det_res.boxes is not None else []

        seg_res = seg_model(img, verbose=False, conf=args.conf)[0]
        seg_masks = []
        if seg_res.masks is not None and seg_res.boxes is not None:
            mask_data = seg_res.masks.data.cpu().numpy()
            for i, c in enumerate(seg_res.boxes.cls.cpu().numpy()):
                if int(c) == args.trunk_class_id:
                    seg_masks.append(cv2.resize(mask_data[i], (w, h),
                                     interpolation=cv2.INTER_NEAREST).astype(np.uint8))

        det_ious, seg_ious = [], []
        for gt_pts, gt_box in gt_instances:
            gt_mask = polygon_to_mask(gt_pts, w, h)
            det_ious.append(max([box_iou(db, gt_box) for db in det_boxes], default=0.0))
            seg_ious.append(max([mask_iou(sm, gt_mask) for sm in seg_masks], default=0.0))

        rows.append({
            "frame": stem,
            "blur_score": blur_score,
            "brightness_mean": brightness_mean,
            "n_instances": len(gt_instances),
            "mean_det_iou": np.mean(det_ious),
            "mean_seg_iou": np.mean(seg_ious),
        })

    df = pd.DataFrame(rows)
    df.to_csv(args.out_csv, index=False)

    df["blur_group"] = pd.qcut(df["blur_score"], 3,
        labels=["High Blur (Low Variance)", "Medium Blur", "Low Blur (High Variance)"])
    df["lighting_group"] = pd.qcut(df["brightness_mean"], 3,
        labels=["Low Brightness", "Medium Brightness", "High Brightness"])

    print(f"\nEvaluated {len(df)} frames.\n")

    print("=== TC-12: Robustness by Motion Blur Group ===")
    blur = df.groupby("blur_group", observed=True)[["mean_det_iou", "mean_seg_iou"]].mean()
    blur["n_frames"] = df.groupby("blur_group", observed=True).size()
    print(blur.to_string())

    print("\n=== TC-13: Robustness by Lighting Group ===")
    light = df.groupby("lighting_group", observed=True)[["mean_det_iou", "mean_seg_iou"]].mean()
    light["n_frames"] = df.groupby("lighting_group", observed=True).size()
    print(light.to_string())

    print(f"\nSaved per-frame results to {args.out_csv}")


if __name__ == "__main__":
    main()

In [ ]:
# To run robustness evaluation (RQ3)
!python robustness_stratify.py \
  --det_model "/content/drive/MyDrive/Colab Notebooks/best.pt/trunk_det/best_trunkdet.pt" \
  --seg_model "/content/drive/MyDrive/Colab Notebooks/best.pt/trunk_seg/bestseg.pt" \
  --images "/content/gt_check/images/Train" \
  --labels "/content/gt_check/labels/Train" \
  --trunk_class_id 0 \
  --out_csv /content/evaluation_outputs/robustness_results.csv

In [ ]:
# ===== CELL 9: Write domain_gap_confidence.py =====
%%writefile domain_gap_confidence.py
""" Domain gap indicator for the weed model.

No ground-truth weed labels exist for the plantation video, so this compares
model BEHAVIOUR (confidence + detection rate) between:
  (a) the external dataset's own validation split  - in-domain
  (b) the actual plantation video                  - out-of-domain

A large confidence drop and a rise in zero-detection frames is indirect
evidence that learned features do not transfer across domains. """

import argparse, glob, os
import numpy as np
import pandas as pd
from ultralytics import YOLO


def analyze_images(model, image_folder, conf_thresh):
    all_confs, det_counts = [], []
    paths = sorted(glob.glob(os.path.join(image_folder, "*.*")))
    for p in paths:
        res = model(p, verbose=False, conf=conf_thresh)[0]
        n = len(res.boxes) if res.boxes is not None else 0
        det_counts.append(n)
        if n:
            all_confs.extend(res.boxes.conf.cpu().numpy().tolist())
    return all_confs, det_counts, len(paths)


def analyze_video(model, video_path, conf_thresh):
    all_confs, det_counts, n_frames = [], [], 0
    for res in model(video_path, verbose=False, conf=conf_thresh, stream=True):
        n_frames += 1
        n = len(res.boxes) if res.boxes is not None else 0
        det_counts.append(n)
        if n:
            all_confs.extend(res.boxes.conf.cpu().numpy().tolist())
    return all_confs, det_counts, n_frames


def summarize(name, confs, det_counts, n_units):
    zero_frac = sum(1 for c in det_counts if c == 0) / len(det_counts) if det_counts else 0
    print(f"\n--- {name} ---")
    print(f"Images/frames processed         : {n_units}")
    print(f"Total detections                : {len(confs)}")
    print(f"Mean confidence                 : {np.mean(confs):.4f}" if confs else "Mean confidence: N/A")
    print(f"Median confidence               : {np.median(confs):.4f}" if confs else "")
    print(f"Mean detections per image/frame : {np.mean(det_counts):.3f}")
    print(f"Fraction with zero detections   : {zero_frac:.1%}")
    return {
        "source": name,
        "n_units": n_units,
        "total_detections": len(confs),
        "mean_confidence": np.mean(confs) if confs else None,
        "mean_detections_per_unit": np.mean(det_counts),
        "zero_detection_fraction": zero_frac,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weed_model", required=True)
    ap.add_argument("--val_images", required=True)
    ap.add_argument("--video", required=True)
    ap.add_argument("--conf", type=float, default=0.25)
    ap.add_argument("--out_csv", default="domain_gap_confidence.csv")
    args = ap.parse_args()

    model = YOLO(args.weed_model)

    r1 = summarize("External Validation Set (in-domain)",
                   *analyze_images(model, args.val_images, args.conf))

    if os.path.isdir(args.video):
        r2 = summarize("Actual Plantation Video (out-of-domain)",
                       *analyze_images(model, args.video, args.conf))
    else:
        r2 = summarize("Actual Plantation Video (out-of-domain)",
                       *analyze_video(model, args.video, args.conf))

    pd.DataFrame([r1, r2]).to_csv(args.out_csv, index=False)

    print("\n== Domain Gap Indicator Summary ==")
    if r1["mean_confidence"] and r2["mean_confidence"]:
        drop = (r1["mean_confidence"] - r2["mean_confidence"]) / r1["mean_confidence"] * 100
        print(f"Mean confidence drop (external -> video): {drop:.1f}%")
    print(f"Detections per unit: external={r1['mean_detections_per_unit']:.3f}, "
          f"video={r2['mean_detections_per_unit']:.3f}")
    print(f"Zero-detection fraction: external={r1['zero_detection_fraction']:.1%}, "
          f"video={r2['zero_detection_fraction']:.1%}")
    print(f"\nSaved summary to {args.out_csv}")


if __name__ == "__main__":
    main()

In [ ]:
# To run domain gap evaluation (RQ4)
!python domain_gap_confidence.py \
  --weed_model "/content/drive/MyDrive/Colab Notebooks/best.pt/weed_det/best_weeddet.pt" \
  --val_images "/content/drive/MyDrive/weed_yolo_dataset/images/val" \
  --video "/content/drive/MyDrive/Colab Notebooks/fyp_video_sample2.mp4" \
  --out_csv /content/evaluation_outputs/domain_gap_confidence.csv

In [ ]:
# To back up all evaluation outputs to Drive
import shutil

eval_backup = '/content/drive/MyDrive/Colab Notebooks/evaluation_backup'
os.makedirs(eval_backup, exist_ok=True)

# Metric CSVs
for f in glob.glob('/content/evaluation_outputs/*.csv'):
    shutil.copy(f, os.path.join(eval_backup, os.path.basename(f)))
    print("Backed up:", os.path.basename(f))

# Annotated output videos
video_backup = os.path.join(eval_backup, 'inference_videos')
os.makedirs(video_backup, exist_ok=True)
for f in glob.glob('/content/playable_outputs/*.mp4'):
    shutil.copy(f, os.path.join(video_backup, os.path.basename(f)))
    print("Backed up:", os.path.basename(f))

# The evaluation scripts themselves, for reproducibility
for script in ['compute_iou.py', 'robustness_stratify.py', 'domain_gap_confidence.py']:
    if os.path.exists(script):
        shutil.copy(script, os.path.join(eval_backup, script))
        print("Backed up:", script)

print("\nAll evaluation outputs backed up to:", eval_backup)